### Imports

In [12]:
from pathlib import Path
import sys
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import (
    PROJECT_ROOT,
    DATA_DIR,
    RAW_DIR,
    RAW_PDF_DIR,
    BRONZE_DIR,
    SILVER_DIR,
    GOLD_DIR,
    CANONICAL_RDHS,
)

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)


Python: 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
Project root: /media/breezy/NewVolume/projects_int/dengue
Data directory: /media/breezy/NewVolume/projects_int/dengue/data


### check directories

In [13]:
directories = {
    "raw": RAW_DIR,
    "wer_pdf": RAW_PDF_DIR,
    "bronze": BRONZE_DIR,
    "silver": SILVER_DIR,
    "gold": GOLD_DIR,
}

for name, path in directories.items():
    print(f"{name:12} -> {path}| exist={path.exists()}")

raw          -> /media/breezy/NewVolume/projects_int/dengue/data/raw| exist=True
wer_pdf      -> /media/breezy/NewVolume/projects_int/dengue/data/raw/wer_pdfs| exist=True
bronze       -> /media/breezy/NewVolume/projects_int/dengue/data/bronze| exist=True
silver       -> /media/breezy/NewVolume/projects_int/dengue/data/silver| exist=True
gold         -> /media/breezy/NewVolume/projects_int/dengue/data/gold| exist=True


### RDHS check

In [14]:
print("Number of RDHS:", len(CANONICAL_RDHS))

for i,district in enumerate(CANONICAL_RDHS, start=1):
    print(f"{i:02d}. {district}")


Number of RDHS: 26
01. Colombo
02. Gampaha
03. Kalutara
04. Kandy
05. Matale
06. Nuwara Eliya
07. Galle
08. Matara
09. Hambantota
10. Jaffna
11. Kilinochchi
12. Mannar
13. Vavuniya
14. Mullaitivu
15. Batticaloa
16. Ampara
17. Kalmunai
18. Trincomalee
19. Kurunegala
20. Puttalam
21. Anuradhapura
22. Polonnaruwa
23. Badulla
24. Monaragala
25. Ratnapura
26. Kegalle


### Verify 1,015 PDFs already download

In [15]:
pdf_files = sorted(RAW_PDF_DIR.glob("*.pdf"))
print(f"pdf count: {len(pdf_files)}")

pdf count: 1015


In [16]:
for path in pdf_files[:10]:
    print(path.name)

wer_2007_w01.pdf
wer_2007_w02.pdf
wer_2007_w03.pdf
wer_2007_w04.pdf
wer_2007_w05.pdf
wer_2007_w06.pdf
wer_2007_w07.pdf
wer_2007_w08.pdf
wer_2007_w09.pdf
wer_2007_w10.pdf


In [17]:
for path in pdf_files[-10:]:
    print(path.name)

wer_2026_w18.pdf
wer_2026_w19.pdf
wer_2026_w20.pdf
wer_2026_w21.pdf
wer_2026_w22.pdf
wer_2026_w23.pdf
wer_2026_w24.pdf
wer_2026_w25.pdf
wer_2026_w26.pdf
wer_2026_w27.pdf


### check whether filenames look like sensible or not

In [18]:
import re

pattern = re.compile(r"wer_(\d{4})_w(\d{2})\.pdf")

parsed_names = []

for path in pdf_files:
    match = pattern.fullmatch(path.name)
    if match:
        year = int(match.group(1))
        week = int(match.group(2))

        parsed_names.append({
            "file": path.name,
            "year": year,
            "week": week,
        })


name_df = pd.DataFrame(parsed_names)

name_df.head()

,file,year,week
0,wer_2007_w01.pdf,2007,1
1,wer_2007_w02.pdf,2007,2
2,wer_2007_w03.pdf,2007,3
3,wer_2007_w04.pdf,2007,4
4,wer_2007_w05.pdf,2007,5


In [19]:
name_df["year"].value_counts().sort_index()

year
2007    52
2008    52
2009    53
2010    52
2011    52
2012    52
2013    52
2014    52
2015    51
2016    53
2017    52
2018    52
2019    52
2020    52
2021    52
2022    51
2023    52
2024    52
2025    52
2026    27
Name: count, dtype: int64

In [10]:
import re
import datetime
from collections import Counter

pattern = re.compile(r"wer_(\d{4})_w(\d{2})\.pdf")

year_weeks = {}

for path in pdf_files:
    match = pattern.fullmatch(path.name)

    if match:
        year = int(match.group(1))
        week = int(match.group(2))

        year_weeks.setdefault(year, []).append(week)


for year, weeks in sorted(year_weeks.items()):

    existing = set(weeks)

    # Find how many ISO weeks this year should have
    max_week = datetime.date(year, 12, 28).isocalendar().week

    expected = set(range(1, max_week + 1))

    missing = expected - existing
    extra = existing - expected

    duplicates = {
        week: count
        for week, count in Counter(weeks).items()
        if count > 1
    }
    if len(missing) != 0 | len(extra) != 0:
        print(f"\nYear: {year}")
        print(f"Expected weeks: 1-{max_week}")
        print("Missing:", sorted(missing))
        print("Extra:", sorted(extra))
        print("Duplicates:", duplicates)



Year: 2016
Expected weeks: 1-52
Missing: []
Extra: [53]
Duplicates: {}


In [11]:
from src.config import (
    PROJECT_ROOT,
    RAW_PDF_DIR,
    BRONZE_DIR,
    SILVER_DIR,
    GOLD_DIR,
    CANONICAL_RDHS,
)

print("Project:", PROJECT_ROOT)
print("PDF directory:", RAW_PDF_DIR)
print("Number of RDHS:", len(CANONICAL_RDHS))

Project: /media/breezy/NewVolume/projects_int/dengue
PDF directory: /media/breezy/NewVolume/projects_int/dengue/data/raw/wer_pdfs
Number of RDHS: 26
